# 潜空间内梯度匹配与基于 Guidance 的数据集蒸馏

解释一下标题是什么意思。本章介绍的 IGD 尽管理论包装非常豪华，其本质还是潜空间内梯度匹配，并且我个人认为其方法非常复杂且工程化。

更多的，另外两种方法 MGD3 与 DAP 会简洁得多，他们提出基于 Guidance 的数据集蒸馏，即 Diffusion 在去噪过程中始终受到一种额外方向指导。这种方向可以是来自真实数据集各个聚类中心的牵引保证多样化，也可以是来自真实数据集的特征匹配保证代表性。

我们详细来看。

# Influence-Guided Diffusion

值得一提，IGD 疑似没有 arXiv 投稿 https://proceedings.iclr.cc/paper_files/paper/2025/file/d3e50c185ced668857ba40a7d4a29efc-Paper-Conference.pdf Influence-Guided Diffusion for Dataset Distillation 这比较奇怪。

## 基本逻辑

首先，记 $x,y$ 是一条训练样本及其标签，$x',y'$ 是一条用于评估的真实样本及其标签，$\theta_t$ 是分类模型在训练步骤 $t$ 的参数，$\ell(x,y;\theta)$ 表示损失。那么我们知道模型 $\theta_t$ 对于样本 $(x,y)$ 做一步梯度下降结果是
$$\theta_{t+1}-\theta_t=
-\eta_t\nabla_\theta\ell(x,y;\theta_t).$$
我们关心这一步更新如何改变模型对于评估样本 $(x',y')$ 的损失，考虑损失的一阶近似
$$\ell(x',y';\theta_{t+1})-\ell(x',y';\theta_t)\approx
\nabla_\theta\ell(x',y';\theta_t)^\mathsf T
(\theta_{t+1}-\theta_t).$$
代入可以得到
$$\ell(x',y';\theta_{t+1})-\ell(x',y';\theta_t)\approx
-\eta_t
\nabla_\theta\ell(x',y';\theta_t)^\mathsf T
\nabla_\theta\ell(x,y;\theta_t).$$
所以，如果我们有梯度大于零
$$\nabla_\theta\ell(x',y';\theta_t)^\mathsf T
\nabla_\theta\ell(x,y;\theta_t) \gt 0,$$
那么就意味着使用样本 $(x,y)$ 训练一步可以降低模型对于 $(x',y')$ 的损失。

更多的，这个内积越大意味着使用样本 $(x,y)$ 训练对于降低模型对 $(x',y')$ 的损失的帮助越大。

现在我们将这种梯度内积对于各个 epoch 相加，得到
$$\mathcal I(x,x')=
\sum_{e=0}^{E}
\bar\eta_e
\nabla_\theta\ell(x',y';\theta_e)^\mathsf T
\nabla_\theta\ell(x,y;\theta_e).$$
根据前面的一阶近似，实际上这个总影响力会近似正比于减少的损失
$$\mathcal I(x,x')\propto
\ell(x',y';\theta_0)-
\ell(x',y';\theta_E).$$

请注意上述这个估计很粗糙，因为我们实际训练时不会仅仅使用一个样本 $(x,y)$ 而是多个 mini-batch。对于总影响力，一个更合理的解释是我们做梯度内积是考察梯度相似性，换言之思想很接近 Gradient Matching。

总之，我们希望做这样一件事
$$\max_z
\frac{1}{|\mathcal T_c|}
\sum_{i=1}^{|\mathcal T_c|}
\mathcal I\bigl(D(z),x_i^c\bigr).$$
其中 $D(z)$ 是从 $z$ Decoding 之后的合成数据集图像，$\mathcal T_c$ 是真实训练集中属于类别 $c$ 的样本集合并且 $x_i^c\in\mathcal T_c$。

以上目标可以写为
$$\max_z
\sum_{e=0}^{E}
\bar\eta_e
\overline g_{e,c}^{\mathcal S\,\mathsf T}
g_e^{\mathrm{gen}}(z),$$
其中
$$\overline g_{e,c}^{\mathcal S}=
\frac{1}{|\mathcal T_c|}
\sum_{i=1}^{|\mathcal T_c|}
\nabla_\theta
\ell(x_i^c,c;\theta_e^{\mathcal S}),$$

$$g_e^{\mathrm{gen}}(z)=
\nabla_\theta
\ell(D(z),c;\theta_e^{\mathcal S}).$$
此处 $\theta_e^{\mathcal S}$ 是模型在合成数据集 $\mathcal{S}$ 上训练得到的第 $e$ 个 checkpoint。

因此我们实际上在寻找一张合成图，使它在合成数据训练轨迹的每个 checkpoint 上产生的参数梯度，都尽量对齐真实类别的平均梯度。

但是显然的，以上这个目标不可以直接作为优化对象，原因是过于笨重。每当潜空间合成图像 $z$ 进行变动，我们都需要重新计算这个量，计算负非常沉重。更多的，我们直接对所有图像计算平均，非常容易得到模糊的梯度方向，导致类内多样性消失。

现在我们开始指出工程中实际使用的优化对象。首先我们做一个代换
$$\theta_e^{\mathcal S}
\quad\longrightarrow\quad
\theta_e^{\mathcal T}.$$
在理想情况下，若合成数据真能复现真实训练动力学，那么两者在对应 checkpoint 上应产生相同梯度，所以这个替换具有合理性。更多的，这个代换允许我们事先离线训练一系列基于真实数据集的 checkpoints，这会大大减少计算负担。

另外一件事是，我们将梯度内积改为余弦相似度计算。这是合理的，因为正如我们上述所说，我们只是希望得到一个梯度相似性。最终的优化对象是
$$\mathcal G_I(z)=
\frac{1}{|\mathcal R|}
\sum_{e\in\mathcal R}
\bar\eta_e
\left(
1-
\frac{
\overline g_{e,c}^{\mathcal T\,\mathsf T}
g_e^{\mathrm{gen}}(z)
}{
\left\|\overline g_{e,c}^{\mathcal T}\right\|_2
\left\|g_e^{\mathrm{gen}}(z)\right\|_2
}
\right).$$
其中 $\mathcal R$ 是被保存的具有代表性 checkpoints，$\overline g_{e,c}^{\mathcal T}$ 是真实类别 $c$ 在 checkpoint $\theta_e^{\mathcal T}$ 上的平均梯度，$g_e^{\mathrm{gen}}(z)$ 是候选图像 $D(z)$ 在同一 checkpoint 上产生的梯度。

$\mathcal G_I(z)$ 越小表示梯度方向越一致。

我们回顾一下 Gradient Matching 的做法，会发现他们非常相似
$$\min_\mathcal{S} \mathbb{E}_{\theta_0 \sim P_{\theta_0}} \Big[ \sum_{t=0}^{T-1} D(\nabla_\theta \mathcal{L}^\mathcal{S}(\theta_t), \nabla_\theta \mathcal{L}^\mathcal{T}(\theta_t)) \Big]$$

更多的，对于潜空间元素 $z$ 也很有说法。首先
$$g_e^{\mathrm{gen}}(z)=
\nabla_\theta
\ell(D(z),c;\theta_e^{\mathcal T}).$$
这个式子不一定有意义，因为直接对原始 $z$ Decoding 可能会得到意义极低的图像，让判别器模型去做损失很容易引入错误梯度。所以我们首先将潜空间元素 $z$ 推向真实图像
$$\widehat z_{0\mid t}=
\frac{
z_t-
\sqrt{1-\alpha_t}\,
\epsilon_\phi(z_t,t,c)
}{
\sqrt{\alpha_t}
}.$$
然后考虑优化 $\mathcal G_I(\widehat z_{0\mid t})$ 而不是 $\mathcal G_I(z)$。

其次的，作者给出了挑选 $z$ 的方法。我们记 $\mathcal M^c$ 是类别 $c$ 已经生成的潜空间合成数据集，$z$ 是当前候选潜空间合成元素，$z^*$ 是当前潜空间合成数据集中与 $z$ 最邻近的潜空间合成元素。具体来说
$$z^*=
\arg\max_{\widetilde z\in\mathcal M^c}
\frac{
z^\mathsf T\widetilde z
}{
\|z\|_2\|\widetilde z\|_2
}.$$
我们计算相似度
$$\mathcal G_D(z)=
\frac{
z^\mathsf Tz^*
}{
\|z\|_2\|z^*\|_2
}.$$
现在更新 $z$，朝向
$$- \nabla_z\mathcal G_D(z),$$
这意味着我们采样的 $z$ 永远距离已有潜空间合成数据集比较远，这会保证多样性。

所以，实际上更新潜空间候选元素 $z$ 会受到以上两种方法共同影响。原始 LDM 一步去噪是
$$z_{t-1}=
s\bigl(z_t,t,\epsilon_\phi\bigr),$$
其中 $s$ 是采样器算法。现在我们改为
$$z_{t-1}=
s\bigl(z_t,t,\epsilon_\phi\bigr)
-
\rho_t
\nabla_{z_t}
\mathcal G_I(\widehat z_{0\mid t})
-
\gamma_t
\nabla_{z_t}
\mathcal G_D(z_t).$$
各个部分分别是说，我们希望一步去噪结果可以具有真实图像特征，对于真实数据集拥有类似梯度产生，并且远离现有合成数据集。

上述公式中
$$\rho_t=
k\sqrt{1-\alpha_t}
\frac{
\|\epsilon_\phi(z_t,t,c)\|_2
}{
\left\|
\nabla_{z_t}\mathcal G_I(\widehat z_{0\mid t})
\right\|_2
}.$$
其中 $k$ 是一个权重，$\epsilon_\phi(z_t,t,c)$ 是去噪模型预测结果，这是为了保证 influence guidance 的实际范数大致与当前 denoising signal 的范数成比例，否则很容易会使得梯度向一边倾斜。

所以肉眼可见，IGD 是个比较复杂的蒸馏算法。我们下面重新梳理整个算法流程。

## 蒸馏算法

首先我们提前准备 Diffusion 模型 $\epsilon_\phi$ 与配套的 VAE Decoder $D$，有代表性 checkpoints $\mathcal R$，每个 checkpoint 对于类别 $c$ 的平均梯度列表 $\mathcal G_c$，已有的潜空间合成数据集 $\mathcal M_c$，一个调度时间区间 $[A,B]$。

首先采样高斯噪声 
$$z_T\sim\mathcal N(0,I).$$
对于每个时间步 $t=T,T-1,\ldots,1$，如果时间步 $t \in [A,B]$，我们使用完整的更新公式，否则仅仅使用常规去噪采样器。

最终合成数据是 $x_{\mathrm{syn}}=
D(z_0)$，我们将 $z_0$ 加入 $\mathcal M_c$，开始下一轮挑选。

完整算法如下。

$$
\begin{array}{l}
\hline
\textbf{Algorithm 1:} \text{ Influence-Guided Diffusion Sampling} \\
\hline
\textbf{1 } \textbf{Parameters:} \text{ Class } c, \text{ influence factor } \rho_t, \text{ deviation factor } \gamma_t, \text{ scales } \{\alpha_t\}_{t=1}^T, \text{ guided range } A, B \\
\textbf{2 } \textbf{Required:} \text{ Pre-trained diffusion model } \boldsymbol{\epsilon}_{\phi}, \text{ list of retained checkpoints } \mathcal{R}, \text{ list of averaged} \\
\quad \text{gradients } G_c, \text{ generated data memory } \mathcal{M}_c, \text{ decoder model } D \\
\textbf{3 } \textbf{Initialize:} \text{ Sample initial random noise } \mathbf{z}_T \sim \mathcal{N}(0, I); \\
\begin{aligned}
\textbf{4 } & \textbf{for } t = T \textbf{ to } 1 \textbf{ do} \\
\textbf{5 } & \quad \text{Obtain the denoised signal } \boldsymbol{\epsilon}_{\phi}(\mathbf{z}_t, t, c) \text{ from the diffusion model;} \\
\textbf{6 } & \quad \textbf{if } t \text{ in } [A, B] \textbf{ then} \\
\textbf{7 } & \quad\quad \text{Calculate the influence metric } \mathcal{G}_I(\hat{\mathbf{z}}_{0 \mid t}) \text{ as Equation } 

\mathcal G_I(z)=
\frac{1}{|\mathcal R|}
\sum_{e\in\mathcal R}
\bar\eta_e
\left(
1-
\frac{
\overline g_{e,c}^{\mathcal T\,\mathsf T}
g_e^{\mathrm{gen}}(z)
}{
\left\|\overline g_{e,c}^{\mathcal T}\right\|_2
\left\|g_e^{\mathrm{gen}}(z)\right\|_2
}
\right).\\

\text{with } \mathcal{R} \text{ and } G_c; \\
\textbf{8 } & \quad\quad \text{Calculate the deviation metric } \mathcal{G}_D(\mathbf{z}_t) \text{ as Equation } 

z^*=
\arg\max_{\widetilde z\in\mathcal M^c}
\frac{
z^\mathsf T\widetilde z
}{
\|z\|_2\|\widetilde z\|_2
}, \text{ }

\mathcal G_D(z)=
\frac{
z^\mathsf Tz^*
}{
\|z\|_2\|z^*\|_2
}. \\



\text{with } \mathcal{M}_c; \\
\textbf{9 } & \quad\quad \text{Implement guided sampling } \mathbf{z}_{t-1} = s(\mathbf{z}_t, t, \boldsymbol{\epsilon}_{\phi}) - \rho_t \nabla_{\mathbf{z}_t}\mathcal{G}_I(\hat{\mathbf{z}}_{0 \mid t}) - \gamma_t \nabla_{\mathbf{z}_t}\mathcal{G}_D(\mathbf{z}_t); \\
\textbf{10} & \quad \textbf{else} \\
\textbf{11} & \quad\quad \text{Implement vanilla sampling } \mathbf{z}_{t-1} = s(\mathbf{z}_t, t, \boldsymbol{\epsilon}_{\phi}); \\
& \textbf{for end}
\end{aligned} \\
\textbf{12} \textbf{ return} \text{ Decoded synthetic image } D(\mathbf{z}_0); \\
\hline
\end{array}
$$

我们讨论一下如何挑选有代表性的 checkpoints。这关于到
$$\mathcal G_I(x)=
\frac{1}{|\mathcal R|}
\sum_{e\in\mathcal R}
\bar\eta_e
\left(
1-
\cos
\left(
g_{e,c}^{\mathcal T},
g_e(x)
\right)
\right).$$
朴素策略是每隔固定间隔 epoch 就挑选一个 checkpoint，这显然是有待优化的，因为不同 epoch 中训练的梯度改变剧烈程度不一样。

对于类别 $c$ 我们定义第 $e$ 个 epoch 检查点类平均梯度是
$$g_{e,c}^{\mathcal T}=
\mathbb E_{x_i^c\in\mathcal T_c}
\left[
\nabla_\theta
\ell
\left(
x_i^c,c;\theta_e^{\mathcal T}
\right)
\right].$$

初始仅仅保存第一个检查点
$$\mathcal R=
\left\{
\theta_0^{\mathcal T}
\right\}.$$
并且我们参考其平均梯度
$$g_{\mathrm{ref}}=
g_{0,c}^{\mathcal T}.$$
对于后续每个 checkpoint $\theta_e^{\mathcal T}$，计算
$$s_e=
\frac{
\left(g_{e,c}^{\mathcal T}\right)^{\mathsf T}
g_{\mathrm{ref}}
}{
\left\|g_{e,c}^{\mathcal T}\right\|_2
\left\|g_{\mathrm{ref}}\right\|_2
}.$$
如果
$$s_e \lt \tau,$$
我们认为训练方向出现较大变化，于是更新 checkpoints 集合与参考梯度
$$\mathcal R\leftarrow
\mathcal R\cup
\left\{
\theta_e^{\mathcal T}
\right\},$$
$$g_{\mathrm{ref}}\leftarrow
g_{e,c}^{\mathcal T}.$$
因此我们最终是按照平均梯度变化间隔选取 checkpoints。原文 $\tau=0.7$。

所以最终保存检查点趋势是，初期检查点保存很密集，后期比较稀疏。

## 成果与讨论

IGD 超越了过去的 State-of-the-Art，比如在 ImageNet-1K IPC为 50 与 10 的配置下超越了 RDED 方法。这证明了其方法的有效性。

但是我个人的看法是，IGD 是一篇过渡性成果。他们没有提出特别具有直觉性的内容。如果快速概括，IGD 是一个加上一些工程技巧的 LD3M 与 GM 方法。更多的，实际上 IGD 方法的计算量并不小，尤其是影响力梯度的计算涉及一个二阶梯度，这是加重计算负担的。

我们快速来看下一篇，MGD3。

# MGD3

一般的 Diffusion 模型在生成图像时，非常容易生成最常见最普遍的几个图案样式。所以当我们使用 Diffusion 进行数据集蒸馏，一个难题是多样性的创造。我们认为，蒸馏数据集的重点问题不是图片不真实，而是 Diffusion 生成没有明确覆盖类内模式。

MGD3 原文是 https://arxiv.org/pdf/2505.18963v1 MGD3: Mode-Guided Dataset Distillation using Diffusion Models 我们详细说。

## 基本逻辑

我们将数据集蒸馏分为三个阶段，首先是 Mode Discovery。我们记类别 $c$ 真实图像集合为
$$\mathcal T_c=
\left\{
x_1^{(c)},x_2^{(c)},\ldots,x_{n_c}^{(c)}
\right\},$$
我们使用 VAE Encoder 将每张图编码到潜空间
$$z_j^{(c)}=
\mathcal E\left(x_j^{(c)}\right).$$
现在我们对潜空间元素做 K-means
$$\mathcal M_c=
\left\{
m_1^{(c)},m_2^{(c)},\ldots,m_K^{(c)}
\right\},$$

第二阶段被称为 Mode Guidance。首先采样高斯噪声
$$x_T\sim\mathcal N(0,I),$$
Diffusion 模型可以根据带噪图像与时间步 $t$ 预测干净图像
$$\widehat x_0^{\,t}.$$
现在我们计算这个量
$$g_t=
m_i^{(c)}-\widehat x_0^{\,t},$$
其中 $m_i^{(c)}$ 是当前指定的聚类方向。这个量是一个朴素的当前预测指向聚类中心的方向。


现在我们将这个方向量加入原始 Diffusion 去噪采样器
$$\widehat\epsilon_\theta(x_t,t,c)=
\widetilde\epsilon_\theta(x_t,t,c)
+
\lambda\sigma_t g_t,$$
此处 $\widetilde\epsilon_\theta(x_t,t,c)$ 是 CFG 之后原始噪声预测，$\lambda$ 是 Mode Guidance 权重，$\sigma_t$ 是噪声强度调度。

我们使用 $\widehat\epsilon_\theta(x_t,t,c)$ 来做去噪，最终会得到一个真实图像。但是这还不够，我们指出 Mode Guidance 权重调度的一个细节，也就是最后一个阶段 Stop Guidance。

当 $t$ 下降到 $t_{\mathrm{stop}}$ 之后，我们令
$$\lambda=0.$$
也就是说恢复为普通采样
$$\widehat\epsilon_\theta(x_t,t,c)=
\widetilde\epsilon_\theta(x_t,t,c).$$
为什么这样做？我们认为聚类中心不一定在真实流形上，因此一直向聚类中心引导很有可能走向不真实的生成。在生成后期，我们让模型自行寻找最近的真实图像。

关于聚类中心的指定，实际上非常简单，我们令聚类数量就是 IPC，那么我们每次进行一次生成指定一个聚类即可
$$x_T^{(1)}
\longrightarrow
m_1^{(c)},$$
$$x_T^{(2)}
\longrightarrow
m_2^{(c)},$$
$$\cdots$$

所以 MGD3 方法是一个显然比 IGD 简洁得多的方法。更多的，MGD3 思想上很像 D4M。我们都是对于原始真实图像在潜空间内做 K-means，但是 D4M 与 MGD3 使用 K-means 方法不同，前者将聚类中心加噪之后作为生成起点，后者将聚类中心作为 Guidance 方向。

下面这张图详细展示了 MGD3 在做什么。我们发现每个类的聚类中心，之后作为参考进行更新。

<img src="./assets/MGD3.png" width="900" height="500">

更多的，下面这张图展示了 MGD3 与其他方法区别。这里解释一下什么是 MinMax，我们后续会详细说这个方法。MinMax 核心思想是微调 Diffusion 使其生成更加多样。我们认为 MinMax 虽然经过更多样的生成微调，但是这种机制没有直接给出聚类中心 Gudiance 强烈。

<img src="./assets/MGD3comp.png" width="900" height="380">

关于 Label，实际上我们已经非常熟悉 Soft Label 方法，这个技巧是与蒸馏方法完全解耦的。对于 MGD3 生成的图像也可以做数据增强区域 Soft Label。

## 蒸馏算法

以下是完整算法，比较简单。

$$
\begin{array}{l}
\hline
\textbf{Algorithm 1 } \text{Mode Guidance with DDIM sampling, given} \\
\text{a diffusion model } \epsilon_{\theta}(x_t), \text{ an estimated mode } m_k \text{ and mode} \\
\text{guidance scale } \lambda. \\
\hline
\textbf{Input: } \text{estimated mode } m_k \text{ and mode guidance scale } \lambda \\
x_T \leftarrow \text{sample from } \mathcal{N}(0, \mathbf{I}) \\
\begin{aligned}
& \textbf{for all } t \text{ from } T \text{ to } 1 \textbf{ do} \\
& \quad \mathbf{g}_t = (m_i - \hat{x}_0^t) \\
& \quad \hat{\epsilon} \leftarrow \epsilon_{\theta}(x_t) - \sqrt{1 - \bar{\alpha}_t} \cdot \lambda \cdot \mathbf{g}_t \\
& \quad x_{t-1} \leftarrow \sqrt{\bar{\alpha}_{t-1}} \left( \frac{x_t - \sqrt{1 - \bar{\alpha}_t} \hat{\epsilon}}{\sqrt{\bar{\alpha}_t}} \right) + \sqrt{1 - \bar{\alpha}_{t-1}} \hat{\epsilon} \\
& \textbf{for end}
\end{aligned} \\
\textbf{return: } x_0 \\
\hline
\end{array}
$$

## 成果与讨论

MGD3 方法产生了可以被观察的提升，并且由于其 Training-free 特点，MGD3 可以随意地加入各类方法中。我们观察到 IPC 低时 MGD3 优势并不明显，但是随着 IPC 增加，其超越了之前最佳方法，这是因为 IPC 越大数据集蒸馏过程越需要类内多样性的指定。

更多的，不仅仅是对简单图像生成模型，对于通用 T2I 模型 MGD3 方法也可以被使用并且带来性能提升。但是需要注意，通用 T2I 模型的训练范围并不仅仅是 ImageNet-1K，这是训练域的不匹配带来一些损失。

MGD3 相较 GLaD 等等方法最大优点就是计算负担极小，后者计算算力需求可能是前者几十倍，这是巨大的进步。

天然地，我们会更喜欢 MGD3 或 RDED 这类方法，因为简单有效。

下面我想说说 DAP。在 MGD3 中我们认为具有代表性的元素就是聚类，但是 DAP 提出不一样的看法，我们认为代表元素在 Diffusion 中与真实元素呈现相近的特征。

# DAP

原文是 https://arxiv.org/abs/2510.17421 Diffusion Models as Dataset Distillation Priors 我们终于看到 2026 年的文章了。

## 基本逻辑

原文的表述比较重，实际上方法同样比较简单，我们展开说。

我们认为一个良好的蒸馏数据集拥有三个特点，多样和泛化和具备原始真实数据集重要特征。对于一个训练良好的 Diffusion 模型，其已经学习了真实数据的分布，因此可以带来前两个方面的优势。我们重点补齐代表性的短板。

一个表述是
$$\nabla_x\log p(x\mid R)=\nabla_x\log p(x)+\nabla_x\log p(R\mid x).$$
其中 $x$ 表示样本，$R$ 表示该样本具有代表性，$\nabla_x\log p(x)$ 是原始 Diffusion 估计的 score，$\nabla_x\log p(R\mid x)$ 是我们需要加入的 guidance。所以 DAP 认为
$$\underbrace{\nabla_x\log p(x)}_{\text{diversity 与 generalization}}
+
\underbrace{\nabla_x\log p(R\mid x)}_{\text{representativeness}}.$$

关于代表性，我们希望优化
$$E\left(x^{\mathrm{syn}}\right)=
\frac{1}{N_c}
\sum_{i=1}^{N_c}
d\left(
\phi(x^{\mathrm{syn}}),
\phi(x_i^{\mathrm{train},c})
\right).$$
其中 $x_i^{\mathrm{train},c}$ 是真实数据类别 $c$ 的第 $i$ 个样本，$\phi$ 是预训练 Diffusion 某一层特征映射，$d$ 是一个度量。

关于 $d$ 这个度量，尽管 DAP 原文写了许多关于 RKHS 的内容，在最后还是选择了 L2 度量。他们的原意应该是希望测试更多的核诱导的度量。

那么 Guidance 如何加入去噪过程？我们记 $x_t^{\mathrm{syn}}$ 表示当前合成元素，$x_{t,i}^{\mathrm{train},c}$ 表示真实图像加噪到时间步 $t$ 时状态，$\phi_t$ 表示来自 Diffusion 网络在时间步 $t$ 下的特征映射。首先计算
$$z_t^{\mathrm{syn}}=
\phi_t\left(x_t^{\mathrm{syn}}\right).$$
$$z_{t,i}^{\mathrm{train},c}=
\phi_t\left(x_{t,i}^{\mathrm{train},c}\right).$$
我们将特征进行比较
$$E_t=
\frac{1}{N_c}
\sum_{i=1}^{N_c}
d\left(
z_t^{\mathrm{syn}},
z_{t,i}^{\mathrm{train},c}
\right).$$
求梯度得到方向
$$g_t=
-\nabla_{x_t^{\mathrm{syn}}}E_t.$$
假设一般采样器给出去噪结果 $\widetilde x_{t-1}$，我们加上 Guidance
$$x_{t-1}=
\widetilde x_{t-1}
+
\gamma g_t.$$

所以 DAP 遵循了与 MGD3 非常类似的思想，我们为去噪过程加上直接的 Guidance 获得一些收益。因此同样的，我们需要 Stop Guidance 保证最终去噪结果在真实数据上。在 $50$ 步数采样重，设置
$$t_{\mathrm{stop}}=25.$$
在 $t > t_{\mathrm{stop}}$ 的时间内，我们停止 Guidance。

## 蒸馏算法

以下是完整算法。注意一点是，$\epsilon_\theta(x_t,t)$ 实际上指的不是噪声预测而是 score 预测，两者之间相差一个噪声调度系数。我无法理解 DAP 原文这样书写的理由，但是我们还是保持一致。 

更多的，Mercer Kernel 诱导的度量就是 L2 度量。

$$
\begin{array}{l}
\hline
\textbf{Algorithm 1 } \text{DAP Sampling (VP-SDE)} \\
\hline
\textbf{Require: } \text{Noisy data samples } \mathbf{x}_t^{\text{train}\mid c} \text{ within class } c, \text{ pre-trained diffusion model } \boldsymbol{\epsilon}_{\theta}, \text{ a layer output} \\
\quad \color{#D2691E}{\phi} \color{black}\text{ from diffusion backbone network, a Mercer Kernel induced distance measurement } d, \text{ energy-} \\
\quad \text{based guidance scale } \gamma, \text{ pre-defined noise scales } \beta_t, \text{ truncation step } t_{\mathrm{stop}}. \\
\begin{aligned}
1: & \ \mathbf{x}_T \sim \mathcal{N}(0, I) \\
2: & \ \textbf{for } t = T, \dots, 1 \textbf{ do} \\
3: & \ \quad \boldsymbol{\epsilon} \sim \mathcal{N}(0, I) \textbf{ if } t > 1, \textbf{ else } \boldsymbol{\epsilon} = \mathbf{0} \\
4: & \ \quad \tilde{\mathbf{x}}_{t-1} = (2 - \sqrt{1 - \beta_t})\mathbf{x}_t + \beta_t \boldsymbol{\epsilon}_{\theta}(\mathbf{x}_t, t) + \sqrt{\beta_t}\boldsymbol{\epsilon} \\
5: & \ \quad \mathbf{z}_t = \color{#D2691E}{\phi}\color{black}(\mathbf{x}_t), \mathbf{z}_t^{\text{train}\mid c} = \color{#D2691E}{\phi}\color{black}(\mathbf{x}_t^{\text{train}\mid c}) \quad \quad \quad \quad \quad \quad \quad \, \text{\# Diffusion as representativeness priors} \\
& \ \quad \textbf{if } t > t_{\mathrm{stop}} \textbf{ then} \\
6: & \ \quad\quad \mathbf{g}_t = -\nabla_{\mathbf{x}_t} d(\mathbf{z}_t, \mathbf{z}_t^{\text{train}\mid c}) \\
7: & \ \quad\quad \mathbf{x}_{t-1} = \tilde{\mathbf{x}}_{t-1} + \gamma \mathbf{g}_t \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \, \text{\# Guided sampling} \\
& \ \quad \textbf{else} \\
8: & \ \quad\quad \mathbf{x}_{t-1} = \tilde{\mathbf{x}}_{t-1} \\

9: & \ \textbf{for end}
\end{aligned} \\
\textbf{Output: } x_0 \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \, \text{\# The distilled sample of class } c. \\
\hline
\end{array}
$$

## 成果与讨论

我想重点讨论一下关于 DAP 的多样性问题。实际上 DAP 多样性保证远远不如 MGD3 那么强烈，后者通过 K-means 天然承认了数据集多峰分布的存在，但是前者的多样性仅仅来自初始随机采样高斯噪声与特征比较的真实数据样本采样情况。在不理想的情况下，DAP 的多样性很有可能坍塌。

更有趣的一点是，MGD3 的出发点是 Diffusion 模型生成图案天然没有多样性，因此采取 K-means 强调多样性的指导。但是 DAP 反过来认为 Diffusion 的采样天然具备多样性的特点，并且强调了蒸馏数据集的代表性，

我个人更加认同 MGD3 的观点，因为他们用可视化证明了原始 DiT 等等模型的多样性采样确实很弱。更多的，MinMax 等等成果也指出 Diffusion 模型生成多样性是需要刻意微调优化的。因此我不认为 DAP 的这个假设成立。

从成果上来看，DAP 方法超过了过去的诸多方法，包括 MGD3 与 IGD。更多的，我们指出一件事，许多方法依靠 Soft Label，但是 DAP 在 Hard Label 下表现依然可观。

从 DAP 的实验结果来看，也许他们是对的，因为 t-SNE 投影下 DAP 的蒸馏数据集呈现了天然的聚类特征。

# 总结

本章介绍了三种方法，IGD 与 MGD3 与 DAP，其中 IGD 做了一种多 checkpoints 的梯度匹配，MGD3 做了潜空间元素之间的聚类，DAP 则做了蒸馏数据与真实数据集之间的特征匹配。以上三种方法均是经过认可的方法，但是 IGD 显然远远比后两者更加复杂。

下一章我想介绍一个更加实用且自然的内容，CoDA。我们提出一种对于 T2I 通用生成模型做数据集蒸馏的普遍方法。